In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    print("Torch version:", torch.__version__)

CUDA available: True
GPU name: NVIDIA A100 80GB PCIe
CUDA version: 12.8
Torch version: 2.8.0+cu128


In [2]:
%%capture
%pip install vllm # note that vllm also installs many dependencies such as transformers, torch, pydantic etc.
%pip install seaborn

In [3]:
!pip install --upgrade huggingface_hub transformers accelerate safetensors sentencepiece

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
from huggingface_hub import login
login("token")

In [2]:
from huggingface_hub import whoami
print(whoami())

{'type': 'user', 'id': '67dfef658fce6274d6a0fe73', 'name': 'Nanchen1', 'fullname': 'Chen', 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/no-auth/s-Scr2y72r-rSTNBYO6wc.png', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'llm1017', 'role': 'fineGrained', 'createdAt': '2025-10-17T15:42:28.210Z', 'fineGrained': {'canReadGatedRepos': True, 'global': [], 'scoped': [{'entity': {'_id': '67dfef658fce6274d6a0fe73', 'type': 'user', 'name': 'Nanchen1'}, 'permissions': ['repo.content.read', 'repo.write']}]}}}}


In [7]:
import pandas as pd
import random

N = 30  # Target number of background profiles to generate

# Gender pool
genders = ["Male", "Female"]

# Region pool grouped by continents (for diversity)
regions = {
    "Europe": ["Germany", "UK", "France", "Italy", "Spain", "Sweden", "Poland"],
    "Asia": ["China", "Japan", "India", "South Korea", "Singapore", "Indonesia"],
    "Americas": ["USA", "Canada", "Brazil", "Mexico", "Argentina"],
    "Africa": ["Nigeria", "Egypt", "South Africa", "Kenya"],
    "Oceania": ["Australia", "New Zealand"]
}

# Occupation pool grouped by broad categories
occupations = {
    "Education": ["Teacher", "University student", "Researcher", "Librarian"],
    "Healthcare": ["Doctor", "Nurse", "Psychologist", "Pharmacist"],
    "Technology": ["Software engineer", "Data analyst", "IT manager", "UX designer"],
    "Business": ["Entrepreneur", "Accountant", "Marketing manager", "Salesperson"],
    "Creative": ["Writer", "Graphic designer", "Musician", "Architect"],
    "Public Service": ["Civil servant", "Social worker", "Lawyer", "Police officer"]
}

# Function to generate a random age with weighted distribution
def random_age():
    """
    Randomly generate an age with a realistic weighted distribution:
    - 20–30 years old: 30% (young adults)
    - 31–50 years old: 50% (middle-aged adults)
    - 51–65 years old: 20% (older adults)
    """
    age_groups = [(20, 30, 0.3), (31, 50, 0.5), (51, 65, 0.2)]
    group = random.choices(age_groups, weights=[g[2] for g in age_groups])[0]
    return random.randint(group[0], group[1])

# Function to generate one random background profile
def random_background(i):
    """
    Randomly sample demographic attributes from predefined pools.
    Returns a dictionary containing:
    - Age
    - Gender
    - Region
    - Occupation
    - Continent
    - Occupation category
    """
    gender = random.choice(genders)
    continent = random.choice(list(regions.keys()))
    region = random.choice(regions[continent])
    occ_category = random.choice(list(occupations.keys()))
    occupation = random.choice(occupations[occ_category])
    age = random_age()

    return {
        "id": i + 1,
        "age": age,
        "gender": gender,
        "region": region,
        "occupation": occupation,
        "continent": continent,
        "occupation_category": occ_category
    }

# Generate N random profiles
backgrounds = [random_background(i) for i in range(N)]
df = pd.DataFrame(backgrounds)

# Remove duplicates (if any)
df = df.drop_duplicates(subset=["age", "gender", "region", "occupation"])
df.to_csv("agent_backgrounds.csv", index=False)
print(f" Saved {len(df)} background profiles to agent_backgrounds.csv")
print(df.head(10))

 Saved 30 background profiles to agent_backgrounds.csv
   id  age  gender        region        occupation continent  \
0   1   60  Female     Singapore     Social worker      Asia   
1   2   50  Female        Canada        Accountant  Americas   
2   3   46    Male  South Africa  Graphic designer    Africa   
3   4   48    Male   New Zealand    Police officer   Oceania   
4   5   24  Female   New Zealand           Teacher   Oceania   
5   6   30  Female        France  Graphic designer    Europe   
6   7   47  Female         Egypt          Musician    Africa   
7   8   25  Female       Nigeria      Psychologist    Africa   
8   9   35    Male        Brazil            Doctor  Americas   
9  10   65    Male   New Zealand           Teacher   Oceania   

  occupation_category  
0      Public Service  
1            Business  
2            Creative  
3      Public Service  
4           Education  
5            Creative  
6            Creative  
7          Healthcare  
8          Healthcare  
